# Hierarchical multi-label binary demo

Topic tree from `topics.csv`; **local binary relevance**: one binary classifier per edge `(parent → child)` at branching nodes. Gold labels from **`news_topics.csv`**: positive iff the article’s assigned topics intersect the **subtree** of `child`.

This notebook uses the leaf-balanced **article pool** (`articles_leaf_*`). It trains **H1 (Root → CCAT/ECAT/GCAT/MCAT)**, then optionally **one level down under `GCAT`**. A later section (**task 1a**) compares **LinearSVC**, **GLMNet-style elastic-net logistic**, and **MaxEnt-style L2 logistic** on **H1 metrics only** (no leaf scores). **LinearSVC** + TF-IDF is used in the main cells unless you swap the factory.

**Working directory:** kernel cwd should be the `Homework 4` folder (same directory as `topics.csv`), or adjust `BASE` below.

In [5]:
from pathlib import Path

from topic_hierarchy import binary_edge_specs, load_topic_tree, summary


def homework4_base() -> Path:
    """Resolve Homework 4 folder (contains topics.csv)."""
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — set cwd to Homework 4 or edit BASE")


BASE = homework4_base()
BASE

PosixPath('/Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4')

## Topic tree summary

In [6]:
tree = load_topic_tree(str(BASE / "topics.csv"))
s = summary(tree)
for k, v in sorted(s.items()):
    print(f"  {k}: {v}")

  max_branching_factor: 23
  max_depth_from_root: 5
  n_binary_edge_classifiers: 103
  n_branching_nodes: 22
  n_leaves: 82
  n_local_classifiers: 22
  n_nodes: 117
  traversal_root: Root


## Binary edge inventory (sample)

Each row is one trainable **binary** model: membership in the child’s subtree. Total count is **`n_binary_edge_classifiers`** in the summary above.

In [7]:
edges = binary_edge_specs(tree)
print(f"Total binary edges: {len(edges)}\n")
for sp in edges[:10]:
    print(f"  {sp.parent!r} -> {sp.child!r}  (depth {sp.depth})")

Total binary edges: 103

  'Root' -> 'CCAT'  (depth 0)
  'Root' -> 'ECAT'  (depth 0)
  'Root' -> 'GCAT'  (depth 0)
  'Root' -> 'MCAT'  (depth 0)
  'CCAT' -> 'C1'  (depth 1)
  'CCAT' -> 'C2'  (depth 1)
  'CCAT' -> 'C3'  (depth 1)
  'CCAT' -> 'C4'  (depth 1)
  'ECAT' -> 'E1'  (depth 1)
  'ECAT' -> 'E2'  (depth 1)


## H1 (Root): four branch classifiers — train / test / per-edge metrics

**Scope:** **first-level** edges under `Root` (CCAT, ECAT, GCAT, MCAT). A **second level under `GCAT`** (all `GCAT → child` binaries) is trained in the next section using the same router and data.

**Gold:** from `news_topics.csv` — for edge `Root → child`, label **1** iff the article has any topic in **subtree(`child`)**.

**Procedure (no k-fold):** fit each binary on the **full training** pool via `router.fit_edge`, then report metrics with **`evaluate_binary_edges_from_parent`** on the **same** fitted models — **train** split and **held-out test** split. That matches the models used by `router.edge_model` and `predict_reached_nodes`.

**Classifier:** **`LinearSVC`** (linear SVM) via `binary_edge_factory(..., estimator=LinearSVC, clf_kw=dict(C=..., dual=False, max_iter=...))`. Use `dual=False` with sparse TF-IDF features.

**Router:** `MultilabelHierarchyRouter.predict_reached_nodes` walks every predicted-positive branch (unary nodes pass through).

In [15]:
from sklearn.svm import LinearSVC

from hierarchical_training_data import make_multilabel_binary_pool_data
from hierarchical_classifier import MultilabelHierarchyRouter, binary_edge_factory
from hierarchical_evaluation import evaluate_binary_edges_from_parent

mldata = make_multilabel_binary_pool_data(base_path=str(BASE))

H1_PARENT = "Root"
h1_children = list(tree.children[H1_PARENT])
print("H1 branches:", h1_children)
print("Articles: train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

edge_factory = binary_edge_factory(
    tfidf_kw=dict(min_df=2, max_features=8000),
    estimator=LinearSVC,
    clf_kw=dict(C=1.0, dual=False, max_iter=8000),
)

router = MultilabelHierarchyRouter(tree, edge_factory)

for child in h1_children:
    Xtr, ytr = mldata.binary_edge_dataset(H1_PARENT, child, "train")
    if len(set(ytr)) < 2:
        print(f"Skip {child}: need 2 classes in training")
        continue
    router.fit_edge(H1_PARENT, child, Xtr, ytr, depth=0)

metrics_train = evaluate_binary_edges_from_parent(router, mldata, H1_PARENT, split="train")
metrics_test = evaluate_binary_edges_from_parent(router, mldata, H1_PARENT, split="test")

print("\n--- Per-edge metrics (router models; full train fit) ---")
print("\nTrain:")
for child in sorted(metrics_train.keys()):
    print(f"\n[{H1_PARENT} → {child}]")
    print(" ", {k: round(v, 4) for k, v in metrics_train[child].items()})

print("\nTest (held-out):")
for child in sorted(metrics_test.keys()):
    print(f"\n[{H1_PARENT} → {child}]")
    print(" ", {k: round(v, 4) for k, v in metrics_test[child].items()})

H1 branches: ['CCAT', 'ECAT', 'GCAT', 'MCAT']
Articles: train 14465 test 3617

--- Per-edge metrics (router models; full train fit) ---

Train:

[Root → CCAT]
  {'accuracy': 0.9697, 'precision': 0.9675, 'recall': 0.9653, 'f1': 0.9664, 'roc_auc': 0.9954, 'avg_precision': 0.9946}

[Root → ECAT]
  {'accuracy': 0.9707, 'precision': 0.9706, 'recall': 0.9339, 'f1': 0.9519, 'roc_auc': 0.996, 'avg_precision': 0.9911}

[Root → GCAT]
  {'accuracy': 0.9811, 'precision': 0.9825, 'recall': 0.9745, 'f1': 0.9785, 'roc_auc': 0.9983, 'avg_precision': 0.9979}

[Root → MCAT]
  {'accuracy': 0.9838, 'precision': 0.9712, 'recall': 0.9115, 'f1': 0.9404, 'roc_auc': 0.998, 'avg_precision': 0.9881}

Test (held-out):

[Root → CCAT]
  {'accuracy': 0.9007, 'precision': 0.8878, 'recall': 0.8873, 'f1': 0.8876, 'roc_auc': 0.9622, 'avg_precision': 0.9526}

[Root → ECAT]
  {'accuracy': 0.9179, 'precision': 0.8829, 'recall': 0.8382, 'f1': 0.86, 'roc_auc': 0.9624, 'avg_precision': 0.9331}

[Root → GCAT]
  {'accuracy': 0.

## Second level under `GCAT`: train / test / per-edge metrics

**Scope:** every branching edge **`GCAT → child`** from `topics.csv` (Government/Social subtopics: GCRIM, GDEF, GPOL, GSCI, …). Same gold rule as H1: **1** iff `news_topics` intersects **subtree(`child`)** on the full article pool.

**Stacking:** uses the **same** `MultilabelHierarchyRouter` and `mldata` as H1 — deeper binaries are independent TF-IDF+SVM models per edge; `predict_reached_nodes` will use them once `GCAT` is on the BFS queue (i.e. after a positive `Root → GCAT` prediction).

**`fit_edge(..., depth=1)`:** parent depth in the tree is 1 (matches `binary_edge_specs` for edges under `Root`’s grandchildren).

In [17]:
LEVEL2_PARENT = "GCAT"
gcat_children = list(tree.children[LEVEL2_PARENT])
print(f"{LEVEL2_PARENT}: {len(gcat_children)} outgoing edges — sample:", gcat_children[:6], "...")

for child in gcat_children:
    Xtr, ytr = mldata.binary_edge_dataset(LEVEL2_PARENT, child, "train")
    if len(set(ytr)) < 2:
        print(f"Skip {LEVEL2_PARENT} → {child}: need 2 classes in train (n={len(ytr)}, pos={sum(ytr)})")
        continue
    router.fit_edge(LEVEL2_PARENT, child, Xtr, ytr, depth=1)
    print("Fit a child")

print("pre metrics")
# metrics_train_g = evaluate_binary_edges_from_parent(router, mldata, LEVEL2_PARENT, split="train")
# print("metrics 1")
metrics_test_g = evaluate_binary_edges_from_parent(router, mldata, LEVEL2_PARENT, split="test")
print("metrics 2")

# print(f"\n--- {LEVEL2_PARENT}: fitted {len(metrics_test_g)} edges with test metrics ---")
# print("\nTrain (subset):")
# for child in sorted(metrics_train_g.keys())[:8]:
#     print(f"  [{LEVEL2_PARENT} → {child}]", {k: round(v, 4) for k, v in metrics_train_g[child].items()})
# if len(metrics_train_g) > 8:
#     print(f"  ... +{len(metrics_train_g) - 8} more")

print("\nTest (held-out, all edges):")
for child in sorted(metrics_test_g.keys()):
    print(f"\n[{LEVEL2_PARENT} → {child}]")
    print(" ", {k: round(v, 4) for k, v in metrics_test_g[child].items()})

GCAT: 23 outgoing edges — sample: ['G1', 'GCRIM', 'GDEF', 'GDIP', 'GDIS', 'GENT'] ...
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Skip GCAT → GMIL: need 2 classes in train (n=14465, pos=0)
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
Fit a child
pre metrics
metrics 2

Test (held-out, all edges):

[GCAT → G1]
  {'accuracy': 0.9798, 'precision': 0.9213, 'recall': 0.8729, 'f1': 0.8965, 'roc_auc': 0.9942, 'avg_precision': 0.9552}

[GCAT → GCRIM]
  {'accuracy': 0.9737, 'precision': 0.8571, 'recall': 0.6571, 'f1': 0.7439, 'roc_auc': 0.9732, 'avg_precision': 0.8199}

[GCAT → GDEF]
  {'accuracy': 0.9881, 'precision': 0.8714, 'recall': 0.6421, 'f1': 0.7394, 'roc_auc': 0.9825, 'avg_precision': 0.8173}

[GCAT → GDIP]
  {'accuracy': 0.9682, 'precision': 0.8, 'recall': 0.5962, 'f1': 0.6832, 'roc_auc': 0.9754, 'avg_precision': 0.7745}

[GC

### Sanity checks (gold labels + per-edge predictions)

**Gold:** `binary_edge_dataset` labels should match recomputing `gold_positive_edge(article_id, child)` for each pool id in order.

**Per-edge:** each H1 binary is independent — compare `predict_binary` on the same document across children.

In [9]:
CHECK_CHILD = "CCAT"
X_tr, y_tr = mldata.binary_edge_dataset(H1_PARENT, CHECK_CHILD, "train")
train_ids = mldata.train_ids()
y_check = [1 if mldata.gold_positive_edge(aid, CHECK_CHILD) else 0 for aid in train_ids]
n_mismatch = sum(1 for a, b in zip(y_tr, y_check) if a != b)
print(
    f"Edge {H1_PARENT} → {CHECK_CHILD}: label mismatches vs recomputed gold (train pool): "
    f"{n_mismatch} / {len(y_tr)}"
)
assert n_mismatch == 0, "binary_edge_dataset must match gold_positive_edge per article"

arts = mldata.articles_df().set_index("id")
sample_tid = mldata.test_ids()[0]
text = arts.loc[sample_tid, "article"]
print(f"\nSample test id={sample_tid}: per-edge predict_binary (Root → child)")
for ch in h1_children:
    m = router.edge_model(H1_PARENT, ch)
    if m is None:
        continue
    print(f"  {ch}: {m.predict_binary(text)}")

Edge Root → CCAT: label mismatches vs recomputed gold (train pool): 0 / 14465

Sample test id=2296: per-edge predict_binary (Root → child)
  CCAT: 1
  ECAT: 1
  GCAT: 1
  MCAT: 0


### Multi-branch inference (sample test articles)

`predict_reached_nodes` vs gold label sets from `news_topics`.

In [10]:
arts = mldata.articles_df().set_index("id")
for tid in mldata.test_ids()[:2]:
    text = arts.loc[tid, "article"]
    pred_nodes = router.predict_reached_nodes(text)
    gold = mldata.article_labels.get(tid, set())
    print("id", tid)
    print("  gold labels (sample):", sorted(gold)[:12], "..." if len(gold) > 12 else "")
    print("  predicted reached nodes (sample):", sorted(pred_nodes)[:12], "..." if len(pred_nodes) > 12 else "")

id 2296
  gold labels (sample): ['C13', 'C31', 'C42', 'CCAT', 'E41', 'ECAT', 'GCAT', 'GJOB'] 
  predicted reached nodes (sample): ['CCAT', 'ECAT', 'GCAT'] 
id 2348
  gold labels (sample): ['C11', 'C31', 'C312', 'CCAT'] 
  predicted reached nodes (sample): ['CCAT'] 


## Task 1a — Model comparison at **H1 (Root)** only: LinearSVC vs GLMNet vs MaxEnt

**Autext-style linear baselines (sklearn):**

| Preset | Role |
|--------|------|
| **LinearSVC** | Linear SVM on TF-IDF |
| **GLMNet_elasticnet** | Elastic-net *logistic* regression (`penalty='elasticnet'`, `solver='saga'`) — common GLMNet analogue |
| **MaxEnt_L2** | L2 *logistic* regression — binary/multinomial max-ent in NLP is typically log-linear (logistic) |

**Metrics (held-out test pool), H1 only:** macro-F1 over the four `Root → child` edges; **pooled micro-F1** over all H1 edge×document pairs; **positive-support–weighted F1** (each edge’s F1 weighted by its positive count on the split).

Leaf / lowest-level metrics are **omitted** here (`include_leaf=False` in `train_and_summarize`) — handle those in a later part of the assignment if required.

Each row retrains **H1 only** (fresh router). Deeper edges (e.g. `GCAT → …`) are not part of this table.

In [19]:
import time

from hierarchical_summary_metrics import (
    comparison_table,
    linear_model_factories,
    train_and_summarize,
)

# Task 1a: H1 (Root) metrics only — no leaf evaluation.
# train_and_summarize fits on the train pool; summary_row(..., split="test") scores on test only.
factories = linear_model_factories(max_features=8000)
n_models = len(factories)
print(
    f"Task 1a: {n_models} models. Fit each on **train**; every number below is **test (held-out)** H1.\n"
    f"Order: {list(factories.keys())}\n",
    flush=True,
)

summary_rows = []
for i, (model_name, factory) in enumerate(factories.items(), start=1):
    print(f"[{i}/{n_models}] {model_name}: fit train → eval test split ...", flush=True)
    t0 = time.perf_counter()
    _, row = train_and_summarize(
        model_name, tree, mldata, factory, split="test", include_leaf=False
    )
    elapsed = time.perf_counter() - t0
    summary_rows.append(row)
    print(
        f"    ok ({elapsed:.1f}s) test: h1_macro_f1={row['h1_macro_f1']:.4f}, "
        f"h1_pooled_micro_f1={row['h1_pooled_micro_f1']:.4f}\n",
        flush=True,
    )

print("Done. Summary table (test metrics only):", flush=True)
summary_df = comparison_table(summary_rows)
num_cols = [c for c in summary_df.columns if c != "model"]
out = summary_df.copy()
out[num_cols] = out[num_cols].round(4)
out

Task 1a: 3 models. Fit each on **train**; every number below is **test (held-out)** H1.
Order: ['LinearSVC', 'GLMNet_elasticnet', 'MaxEnt_L2']

[1/3] LinearSVC: fit train → eval test split ...
    ok (19.0s) test: h1_macro_f1=0.8671, h1_pooled_micro_f1=0.8802

[2/3] GLMNet_elasticnet: fit train → eval test split ...
    ok (35.7s) test: h1_macro_f1=0.8575, h1_pooled_micro_f1=0.8786

[3/3] MaxEnt_L2: fit train → eval test split ...
    ok (16.8s) test: h1_macro_f1=0.8608, h1_pooled_micro_f1=0.8814

Done. Summary table (test metrics only):


,model,h1_macro_f1,h1_pooled_micro_f1,h1_pos_weighted_f1
0,LinearSVC,0.8671,0.8802,0.8798
1,GLMNet_elasticnet,0.8575,0.8786,0.8770
2,MaxEnt_L2,0.8608,0.8814,0.8798


## Leaf-level evaluation (test split)

**How this relates to “cutting” early**

- **Per-edge training labels** already encode subtree membership: for edge `parent → child`, gold is 1 iff the article has *any* topic in `child`’s subtree. Separate per-edge precision/recall (elsewhere) score each binary head on its own.
- **Inference** (`predict_reached_nodes`): at a **branching** parent, you only go down `parent → child` if that edge’s model predicts **1**. If it predicts **0**, you **do not** open that branch — the walk stops exploring that side (your “cut”). Unary edges (only child) are followed automatically — no binary there.
- **Leaf multilabel F1** compares gold **leaf** codes (`news_topics` ∩ taxonomy leaves) to **`predict_reached_leaves`**. If you cut too high, predicted leaves miss gold leaves → **recall** drops; spurious deep positives hurt **precision**.
- **Path-to-gold branching recall** (`path_gold_branch_recall`): for each article, take all **gold leaf** topics; for the union of **branching** edges on paths `Root → … → leaf`, check whether each required edge predicts **1**. A **0** or a **missing** fitted model on such an edge is counted as wrong — that is exactly “should have progressed through this node for this gold leaf but did not.”

**Per-edge FNs are only on edges that should fire:** for each `parent → child`, gold is **1** iff the article has a label in that child’s subtree (`gold_positive_edge`). Predicting **0** on an edge with gold **0** is a true negative, not an FN. So if “none of the children” fire but only **one** subtree was actually required, only that edge’s head gets an FN — not one FN per sibling.

Together: leaf F1 summarizes **leaf set** agreement; path recall scores **routing** on paths to gold leaves; standard per-edge metrics (elsewhere) already handle **should-have-gone-forward** FNs. With only H1 (or partial trees) fitted, path recall and leaf scores will be **pessimistic** until deeper edges exist.

In [ ]:
import time

from hierarchical_summary_metrics import comparison_table, linear_model_factories, train_and_summarize

# Same three linear baselines; adds leaf multilabel F1 + path-to-gold branching recall (test only).
# Slower than task 1a because it scores leaves + paths per model.
factories = linear_model_factories(max_features=8000)
n_models = len(factories)
print(
    f"Leaf + path metrics (test): {n_models} models, fit H1 on train, eval on test.\n",
    flush=True,
)

leaf_rows = []
for i, (model_name, factory) in enumerate(factories.items(), start=1):
    print(f"[{i}/{n_models}] {model_name} ...", flush=True)
    t0 = time.perf_counter()
    _, row = train_and_summarize(
        model_name, tree, mldata, factory, split="test", include_leaf=True
    )
    print(f"    ok ({time.perf_counter() - t0:.1f}s)", flush=True)
    leaf_rows.append(row)

leaf_df = comparison_table(leaf_rows)
num_cols = [c for c in leaf_df.columns if c != "model"]
out_leaf = leaf_df.copy()
out_leaf[num_cols] = out_leaf[num_cols].round(4)
out_leaf